# 在 Notebook 中提问 LightRAG

这个 notebook 用来直接调用项目的 `/chat/stream` 流式接口，不用手写 `curl`。

启动方式：

```bash
python -m agentic_rag.main serve
```

默认 API 地址按 `APP_PORT=8010` 处理，避免和 `.env` 里的 vLLM `8000` 冲突。

In [ ]:
import html
import json
import time
from pathlib import Path
from pprint import pprint

import pandas as pd
import requests
from IPython.display import HTML, display

API_BASE = "http://127.0.0.1:8010"
TRIPLES_PATH = Path("../lightrag/standard_triples.json")

display(HTML("""
<style>
.stream-output-box {
  white-space: pre-wrap;
  overflow-wrap: anywhere;
  word-break: break-word;
  line-height: 1.6;
  font-family: Consolas, 'Courier New', monospace;
}
</style>
"""))


def _iter_sse_events(response):
    buffer = ""
    for chunk in response.iter_content(chunk_size=1, decode_unicode=True):
        if not chunk:
            continue
        buffer += chunk
        while "\n\n" in buffer:
            raw_event, buffer = buffer.split("\n\n", 1)
            for line in raw_event.splitlines():
                if line.startswith("data: "):
                    payload = line[6:].strip()
                    if payload:
                        yield json.loads(payload)


def _render_output(text: str) -> HTML:
    return HTML(
        f'<div class="stream-output-box">{html.escape(text)}</div>'
    )


def ask(question: str, force_route: str = "lightrag", show_timestamps: bool = True):
    payload = {
        "question": question,
        "force_route": force_route,
    }
    output_parts = ["回答:\n\n"]
    answer_parts = []
    route = None
    reason = None
    started_at = time.perf_counter()
    handle = display(_render_output("".join(output_parts)), display_id=True)

    def append(text: str):
        output_parts.append(text)
        handle.update(_render_output("".join(output_parts)))

    with requests.post(
        f"{API_BASE}/chat/stream",
        json=payload,
        stream=True,
        timeout=(10, None),
        headers={"Accept": "text/event-stream"},
    ) as response:
        response.raise_for_status()
        for event in _iter_sse_events(response):
            elapsed = time.perf_counter() - started_at
            prefix = f"[{elapsed:7.2f}s] " if show_timestamps else ""
            event_type = event.get("type")

            if event_type == "ready":
                append(f"{prefix}stream connected\n")
            elif event_type == "meta":
                route = event.get("route")
                reason = event.get("reason")
                append(f"{prefix}路由: {route}\n")
                append(f"{prefix}原因: {reason}\n\n")
            elif event_type == "status":
                message = event.get("message", "")
                append(f"{prefix}[状态] {message}\n")
            elif event_type == "delta":
                chunk = event.get("content", "")
                if chunk:
                    answer_parts.append(chunk)
                    append(chunk)
            elif event_type == "error":
                message = event.get("message") or "流式接口返回错误。"
                append(f"\n{prefix}[错误] {message}\n")
                raise RuntimeError(message)
            elif event_type == "done":
                route = route or event.get("route")
                reason = reason or event.get("reason")
                append(f"\n\n{prefix}[完成] stream finished\n")

    answer = "".join(answer_parts)
    return {
        "answer": answer,
        "route": route,
        "reason": reason,
    }


def load_triples(path: Path = TRIPLES_PATH):
    if not path.exists():
        raise FileNotFoundError(f"未找到标准三元组文件: {path.resolve()}")
    return json.loads(path.read_text(encoding="utf-8"))


def triples_df(path: Path = TRIPLES_PATH):
    triples = load_triples(path)
    return pd.DataFrame(triples)


def search_triples(keyword: str, path: Path = TRIPLES_PATH, limit: int = 20):
    df = triples_df(path)
    lowered = keyword.lower()
    mask = (
        df["subject"].str.lower().str.contains(lowered, na=False)
        | df["predicate"].str.lower().str.contains(lowered, na=False)
        | df["object"].str.lower().str.contains(lowered, na=False)
        | df["evidence"].str.lower().str.contains(lowered, na=False)
    )
    return df.loc[mask].head(limit)


In [ ]:
# 先测一下服务是否正常
requests.get(f"{API_BASE}/health", timeout=30).json()

In [ ]:
ask("Who publishes ASME B16.5-2013?")

In [ ]:
ask("What is Table II-9 about in ASME B16.5-2013?")

In [ ]:
ask("ASME B16.5-2013 如何说明 gasket materials、gasket dimensions 和 ASME B16.21 之间的关系？选择 gasket dimensions 时要考虑哪些 characteristics？")

## 查看标准三元组

重新执行入库后，项目会在 `lightrag/standard_triples.json` 中写入标准谓语三元组。下面几个单元格用来检查导出结果。

In [ ]:
# 查看文件是否存在，以及总共有多少条三元组
triples = load_triples()
print("标准三元组文件:", TRIPLES_PATH.resolve())
print("三元组总数:", len(triples))

In [ ]:
# 随机看前几条
pprint(triples[:5])

In [ ]:
# 以表格形式查看，便于筛选
df = triples_df()
df.head(10)

In [ ]:
# 按关键词筛选，例如查询 ASME B16.5-2013 的相关三元组
search_triples("ASME B16.5-2013")

In [ ]:
# 查询温度和材料相关的标准三元组
search_triples("Graphite")